# Run overview

1. the newest runs with a summary (plan, run kind, sample, status, duration)
2. searching runs by comment / sample / plan
3. everything a run records about itself: the plan call (`plan_request`), code version, roles, derived-value
   definitions, plot hints and the data columns
4. the value history of a device from the dashboard server (`/api/history`) plotted next to a run's own data

In [ ]:
%matplotlib inline
from nbsetup import *
pd.set_option("display.max_colwidth", 60)
plt.rcParams.update({"figure.figsize": (9, 4), "figure.dpi": 100})

## 1. Recent runs

`recent(N)` - newest first, any status (`running` = no stop document: still running, or the process ended
without closing the run).

In [ ]:
N = 30
runs = recent(N)
if runs.empty:
    raise NoRun(f"no runs at all on {TILED_URI}")
runs

In [ ]:
# per plan / kind: how many runs, how many did not end with success, total measuring time
summary = runs.assign(failed=runs["status"] != "success").groupby(["plan", "run_kind"], dropna=False).agg(
    runs=("scan_id", "count"), not_success=("failed", "sum"), total_min=("duration_s", lambda s: round(s.sum() / 60, 1)),
    newest=("scan_id", "max"))
summary.sort_values("newest", ascending=False)

In [ ]:
# timeline: one bar per run (start .. stop), coloured by plan
fin = runs.dropna(subset=["start", "duration_s"])
plans = list(dict.fromkeys(fin["plan"]))
fig, ax = plt.subplots(figsize=(10, 0.35 * len(plans) + 1.5))
for i, p in enumerate(plans):
    part = fin[fin["plan"] == p]
    ax.barh([i] * len(part), pd.to_timedelta(part["duration_s"].clip(lower=5), unit="s"), left=part["start"],
            height=0.6, color=f"C{i % 10}")
ax.set_yticks(range(len(plans)), plans)
ax.set_title(f"the newest {len(fin)} finished runs")
fig.autofmt_xdate(); plt.show()

## 2. Search

- `find_runs(text=...)`: words that must all appear in plan, run kind, comment, sample or scan number
  (the same rule as the dashboard's Runs search), among the newest 300 runs
- exact fields as server-side queries: `find_runs(plan=..., kind=..., sample=..., status=...)`
- Tiled full-text search over all metadata (`tiled.queries.FullText`)
- the dashboard's own search (`/api/runs?q=...`), when the dashboard runs

In [ ]:
TEXT = None      # e.g. "TaS2 delay"; None = the comment of the newest run with one
SAMPLE = None    # None = the sample of the newest run

TEXT = TEXT or next((c for c in runs["comment"] if c), "")
SAMPLE = SAMPLE or runs["sample"].dropna().iloc[0]
words = " ".join(TEXT.split()[:2])
print(f"text {words!r}:")
display(pd.DataFrame([run_row(r) for r in find_runs(text=words, status=None, limit=15)]))
print(f"sample == {SAMPLE!r}:")
display(pd.DataFrame([run_row(r) for r in find_runs(sample=SAMPLE, status=None, limit=15)]))

In [ ]:
from tiled.queries import FullText

word = words.split()[0] if words else "success"
hits = client().search(FullText(word))
print(f"full-text {word!r}: {len(hits)} runs; newest:", [RunView(u, n) for u, n in list(hits.items()[-1:-6:-1])] if len(hits) else [])
try:
    api = dashboard("/api/runs", q=words, limit=10)
    print(f"dashboard search {words!r}:", [(r["scan_id"], r["plan_name"], r["comment"]) for r in api])
except Exception as exc:
    print(f"(skipped) dashboard at {DASHBOARD_URL} not reachable: {type(exc).__name__}")

## 3. What a run records about itself

`RUN = None` takes the newest successful run. Every run carries:

- `plan_request`: the complete plan call (all parameters, defaults included, devices by name) - what
  "Run again" on the dashboard uses
- `code`: the version of all plan + library code; the archive `<data>/code/<version>.zip` holds it
  (`python -m fastq.provenance show <uid>` / `extract <version> <dir>`)
- `roles`: which device played which lab role
- `derived`: the derived-value definitions valid at the start, with the data key of each input
- `plot_hints`: what the plan wants plotted (`default_plots` makes the dashboard's plots from it)

In [ ]:
import json

RUN = None
run = pick_run(RUN, status="success")
print(run)
display(pd.Series(run_row(run)).to_frame("value").T)
req = run.md.get("plan_request") or {}
print("plan call:\n  " + f"{req.get('name', run.md.get('plan_name'))}(" +
      ", ".join(f"{k}={v!r}" for k, v in (req.get("kwargs") or {}).items()) + ")")
print("code:", json.dumps(run.md.get("code"), indent=1))
print("roles:", run.md.get("roles"))
print("derived values:")
for d in run.md.get("derived") or []:
    print("  ", D.describe(d), "  inputs recorded as", d.get("keys"))
print("plot hints:", run.md.get("plot_hints"))
other = {k: v for k, v in run.md.items() if k not in {"plan_request", "code", "roles", "derived", "plot_hints", "hints",
                                                      "uid", "time", "versions"}}
print("other start metadata:", json.dumps(other, indent=1, default=str)[:3000])
print("stop:", {k: run.stop.get(k) for k in ("exit_status", "reason", "num_events")})

In [ ]:
# streams and columns (units, source; derived columns show their formula)
streams = {s: data_keys(run.node, s) for s in run.streams}
cols = pd.DataFrame([{"stream": s, "column": k, "shape": v.get("shape"), "units": v.get("units"),
                      "source": str(v.get("source"))[:70]} for s, dk in streams.items() for k, v in dk.items()])
print({s: (run.stop.get("num_events") or {}).get(s) for s in run.streams})
display(cols.head(40))
print("plots the dashboard shows for this run:")
for p in default_plots(run.md, streams):
    print("  ", p)

## 4. Device history next to a run

The dashboard server records every numeric value of every device (`fastq.history`), so the history before,
during and after a run is available: `GET <FASTQ_URL>/api/history?pv=<PV>&start=<epoch s>&end=<epoch s>`
returns `{t, v}` (raw samples), or per-minute `{t, v, vmin, vmax}` for long spans.

`DEVICE = None` takes the run's temperature device (role `temperature`); the PV is its main (hinted) parameter
from the device's schema, unless `PARAM` names one. The run's own record of that value is drawn on top.

In [ ]:
DEVICE = None
PARAM = None          # schema parameter name, e.g. "Temp"; None = the hinted one
MARGIN_S = 600        # history before / after the run

try:
    DEVICE = DEVICE or (run.md.get("roles") or {}).get("temperature")
    devs = {d["name"]: d for d in dashboard("/api/devices")["devices"]}
    if DEVICE not in devs:
        raise LookupError(f"device {DEVICE!r} not on the dashboard (devices: {sorted(devs)})")
    schema = dashboard(f"/api/schema/{DEVICE}")
    params = [p for p in schema["params"] if p.get("dtype") in ("float", "int") and p.get("readable", True)]
    prm = next(p for p in params if p["name"] == PARAM) if PARAM else \
        next((p for p in params if p.get("kind") == "hinted"), params[0])
    pv = devs[DEVICE]["prefix"] + prm["name"] + ("_RBV" if prm.get("rbv") else "")
    t0, t1 = run.md["time"], run.stop.get("time") or run.md["time"]
    h = dashboard("/api/history", pv=pv, start=t0 - MARGIN_S, end=t1 + MARGIN_S, points=3000)
    print(f"{pv} ({prm.get('doc')}): {len(h['t'])} samples ({h['source']})")
except Exception as exc:
    h = None
    print(f"(skipped) history not available: {type(exc).__name__}: {exc}\n"
          f"the dashboard ({DASHBOARD_URL}, set FASTQ_URL) must run and record the device")

In [ ]:
if h is not None:
    # the run's own column for this value: hinted signal = device name, otherwise <device>_<attr>
    col = DEVICE if prm.get("kind") == "hinted" else f"{DEVICE}_{prm['attr']}"
    srcs = [s for s in run.streams if col in data_keys(run.node, s)]
    fig, ax = plt.subplots()
    if h["t"]:
        ax.plot(to_local(h["t"]), h["v"], "-", lw=1, color="0.4", label=f"history {pv}")
        if h.get("vmin"):
            ax.fill_between(to_local(h["t"]), h["vmin"], h["vmax"], color="0.8", label="min / max per bucket")
    for i, s in enumerate(sorted(srcs, key=lambda s: not s.endswith("_monitor"))):
        t = run.table(s, columns=["time", col])
        ax.plot(to_local(t["time"]), t[col], "." if s.endswith("_monitor") else "o", ms=3 if s.endswith("_monitor") else 5,
                color=f"C{i}", label=f"run {s}")
    ax.axvspan(*to_local([t0, t1]), color="C2", alpha=0.12, label=label(run))
    ax.set_ylabel(f"{DEVICE}.{prm['attr']} ({prm.get('units', '')})"); ax.legend(fontsize=8)
    fig.autofmt_xdate(); plt.show()
    if not srcs:
        print(f"the run did not record {col!r} - only the history is shown")